# Porosity regression on SEM tiles (Porous_Sponge)

Predicts apparent porosity (fraction of pixels at/below the image's Otsu threshold) for 224x224 SEM tiles.

**Before running**
1. Upload `porosity_dataset.zip` (made by `prepare_dataset.py`) to the top level of your Google Drive ("My Drive").
2. In Colab: **Runtime > Change runtime type > T4 GPU**.
3. **Runtime > Run all**. Approve the Google Drive access prompt when it appears.

Results (best model, predictions, plots) are saved to `My Drive/porosity_runs/`.

**What it does**
- Splits ~15% of the *training images* off as validation (by image, near-duplicates kept together). `test.csv` is used once, at the end.
- Fine-tunes an ImageNet-pretrained ResNet-18 with L1 loss. Augmentation is flips and 90° rotations only — brightness/contrast changes would make the threshold-based label wrong.
- Compares the CNN on the test set against two baselines: predicting the mean, and running Otsu on each tile on its own.

In [ ]:
# ---- Settings ----
ZIP_NAME = "porosity_dataset.zip"   # file name in My Drive
VAL_FRACTION = 0.15                 # share of training images used for validation
EPOCHS = 30
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-4
SEED = 42

In [ ]:
import os, sys, zipfile, shutil

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ZIP = "/content/drive/MyDrive/" + ZIP_NAME
    WORK_DIR = "/content/data"
    OUT_DIR = "/content/drive/MyDrive/porosity_runs"
else:  # running locally next to prepare_dataset.py outputs
    DATA_ZIP = os.environ.get("DATA_ZIP", ZIP_NAME)
    WORK_DIR = os.environ.get("WORK_DIR", "data")
    OUT_DIR = os.environ.get("OUT_DIR", "porosity_runs")

assert os.path.exists(DATA_ZIP), f"Can't find {DATA_ZIP} - upload {ZIP_NAME} to My Drive first."
# Unzip onto the local disk: reading many small files straight from Drive is slow.
shutil.rmtree(WORK_DIR, ignore_errors=True)
with zipfile.ZipFile(DATA_ZIP) as zf:
    zf.extractall(WORK_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
print("tiles:", len(os.listdir(os.path.join(WORK_DIR, "tiles"))), "| outputs ->", OUT_DIR)

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import threshold_otsu

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU - enable one in Runtime settings)")

## Data and train / validation split

In [ ]:
images = pd.read_csv(os.path.join(WORK_DIR, "images.csv"))
train_all = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))

# Validation = whole groups of training images (a group = an image plus its near-duplicates),
# so no image contributes tiles to both train and validation.
group_of = images.set_index("source_image")["group"]
train_all["group"] = train_all["source_image"].map(group_of)
groups = sorted(train_all["group"].unique())
rng = np.random.default_rng(SEED)
rng.shuffle(groups)
n_val_images = round(train_all["source_image"].nunique() * VAL_FRACTION)
val_groups, n = set(), 0
for g in groups:
    if n >= n_val_images:
        break
    val_groups.add(g)
    n += train_all.loc[train_all["group"] == g, "source_image"].nunique()

val_df = train_all[train_all["group"].isin(val_groups)].reset_index(drop=True)
train_df = train_all[~train_all["group"].isin(val_groups)].reset_index(drop=True)
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {df['source_image'].nunique():4d} images {len(df):5d} tiles  "
          f"porosity mean {df['porosity'].mean():.3f} std {df['porosity'].std():.3f}")
assert not set(train_df.source_image) & set(val_df.source_image)
assert not set(train_all.source_image) & set(test_df.source_image)

In [ ]:
def load_tiles(df):
    # All tiles fit in memory (~50 KB each), which keeps training fast.
    arr = np.stack([np.asarray(Image.open(os.path.join(WORK_DIR, "tiles", f)).convert("L"))
                    for f in df["tile_filename"]])
    return torch.from_numpy(arr), torch.tensor(df["porosity"].values, dtype=torch.float32)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class TileDataset(torch.utils.data.Dataset):
    def __init__(self, df, augment):
        self.x, self.y = load_tiles(df)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.x[i].float().div(255).unsqueeze(0)
        if self.augment:
            # Label-preserving only: the porosity of a flipped/rotated tile is unchanged.
            if random.random() < 0.5:
                x = x.flip(2)
            x = torch.rot90(x, random.randint(0, 3), dims=(1, 2))
        x = (x.expand(3, -1, -1) - MEAN) / STD   # grayscale -> 3 channels for ImageNet weights
        return x, self.y[i]

train_ds = TileDataset(train_df, augment=True)
val_ds = TileDataset(val_df, augment=False)
test_ds = TileDataset(test_df, augment=False)
loader = lambda ds, shuffle: torch.utils.data.DataLoader(
    ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0, pin_memory=device.type == "cuda")
train_loader, val_loader, test_loader = loader(train_ds, True), loader(val_ds, False), loader(test_ds, False)

## Model and training

In [ ]:
class PorosityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)

    def forward(self, x):
        return torch.sigmoid(self.backbone(x)).squeeze(1)   # porosity is in [0, 1]

@torch.no_grad()
def predict(model, dl):
    model.eval()
    return torch.cat([model(x.to(device)).float().cpu() for x, _ in dl]).numpy()

model = PorosityNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.L1Loss()
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best_path = os.path.join(OUT_DIR, "best_model.pt")
history, best_val = [], float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = loss_fn(model(x).float(), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total += loss.item() * len(y)
    sched.step()
    train_mae = total / len(train_ds)
    val_mae = float(np.abs(predict(model, val_loader) - val_ds.y.numpy()).mean())
    history.append((epoch, train_mae, val_mae))
    if val_mae < best_val:
        best_val = val_mae
        torch.save(model.state_dict(), best_path)
    print(f"epoch {epoch:3d}  train MAE {train_mae:.4f}  val MAE {val_mae:.4f}"
          + ("  *best" if val_mae == best_val else ""))

h = np.array(history)
plt.figure(figsize=(6, 3.5))
plt.plot(h[:, 0], h[:, 1], label="train")
plt.plot(h[:, 0], h[:, 2], label="validation")
plt.xlabel("epoch"); plt.ylabel("MAE"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curve.png"), dpi=120)
plt.show()

## Test set: CNN vs. baselines

Run once, after you are done tuning on validation.
- **Mean**: always predict the training-set mean porosity. Anything useful must clearly beat this.
- **Tile Otsu**: Otsu threshold computed on the tile alone. The labels come from an Otsu threshold on the *whole image*, so this simple rule may already be strong; if the CNN can't beat it, it isn't adding much.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
y_test = test_ds.y.numpy()
pred_cnn = predict(model, test_loader)
pred_mean = np.full_like(y_test, train_df["porosity"].mean())

def tile_otsu(t):
    t = t.numpy()
    if t.min() == t.max():
        return 0.0
    return float((t <= threshold_otsu(t)).mean())
pred_otsu = np.array([tile_otsu(t) for t in test_ds.x])

def metrics(pred):
    err = pred - y_test
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean()),
            "R2": 1 - (err ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()}

results = pd.DataFrame({name: metrics(p) for name, p in
                        [("CNN (ResNet-18)", pred_cnn), ("Tile Otsu", pred_otsu), ("Mean", pred_mean)]}).T
print(f"Test set: {test_df['source_image'].nunique()} images, {len(test_df)} tiles")
display(results.round(4)) if IN_COLAB else print(results.round(4))

out = test_df.assign(pred_cnn=pred_cnn, pred_tile_otsu=pred_otsu)
out.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)
results.to_csv(os.path.join(OUT_DIR, "test_metrics.csv"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, p) in zip(axes, [("CNN (ResNet-18)", pred_cnn), ("Tile Otsu", pred_otsu)]):
    ax.scatter(y_test, p, s=8, alpha=0.5)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="true porosity", ylabel="predicted",
           title=f"{name}: MAE {np.abs(p - y_test).mean():.3f}")
    ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "test_scatter.png"), dpi=120)
plt.show()

## Worst test predictions

Look at these before trusting the numbers. Big errors often point at label problems (gradients, charging) rather than the model.

In [ ]:
worst = np.argsort(-np.abs(pred_cnn - y_test))[:12]
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, i in zip(axes.ravel(), worst):
    ax.imshow(test_ds.x[i], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"true {y_test[i]:.2f} / pred {pred_cnn[i]:.2f}\n{test_df.tile_filename[i][3:13]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "worst_predictions.png"), dpi=100)
plt.show()